# 🔍 OCR Implementation v4 — Qwen2.5-VL-7B-Instruct

| Item | Detail |
|---|---|
| **Model** | `Qwen/Qwen2.5-VL-7B-Instruct` |
| **Why this model?** | Latest Qwen vision-language model (Jan 2025). Specifically trained for document understanding, table OCR, and dense text extraction. Best Qwen model for OCR tasks. |
| **VRAM needed** | ~15 GB in float16 → fits Colab T4 exactly. Use `load_in_4bit=True` (Cell 3 option) if you get OOM. |
| **Languages** | English, Hindi, Chinese, mixed-language documents |
| **Features** | Image OCR · PDF OCR · Keyword Search · Structured Paragraphs · JSON Export · Custom Prompt |

---
### Before running:
```
Runtime → Change runtime type → T4 GPU   ← mandatory
Then: Runtime → Run all
```

## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys

# poppler-utils: required for PDF → image conversion
subprocess.run(["apt-get", "install", "-y", "-q", "poppler-utils"], capture_output=True)

# Python packages
# transformers>=4.52 is required for Qwen2.5-VL support
pkgs = [
    "torch>=2.2.0",
    "torchvision>=0.17.0",
    "transformers>=4.52.0",
    "accelerate>=0.33.0",
    "qwen-vl-utils>=0.0.11",
    "gradio>=5.0.0",
    "Pillow>=10.0.0",
    "pdf2image>=1.17.0",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

# Optional: bitsandbytes for 4-bit quantisation (saves ~8GB VRAM)
# Uncomment the line below if you get CUDA out-of-memory errors
# subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes>=0.43.0"], check=True)

print("✅ All dependencies installed.")

## Cell 2 — Imports

In [ ]:
import re
import traceback

import gradio as gr
import torch
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

# PDF support
try:
    from pdf2image import convert_from_path
    PDF_SUPPORT = True
except ImportError:
    PDF_SUPPORT = False
    print("⚠️  pdf2image not found — PDF uploads disabled. Run Cell 1 first.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32

print(f"torch   : {torch.__version__}")
print(f"device  : {DEVICE}  ({DTYPE})")
print(f"gradio  : {gr.__version__}")
print(f"PDF     : {'enabled' if PDF_SUPPORT else 'disabled'}")

if DEVICE == "cpu":
    print()
    print("⚠️  WARNING: No GPU detected. Qwen2.5-VL-7B will be extremely slow on CPU.")
    print("   Go to Runtime → Change runtime type → T4 GPU and re-run.")

## Cell 3 — Load Qwen2.5-VL-7B-Instruct

> **OOM?** Set `USE_4BIT = True` below to load in 4-bit quantisation (~4 GB VRAM instead of ~15 GB).

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"

# ── Set True if you get CUDA out-of-memory errors ─────────────────────────────
USE_4BIT = False
# ─────────────────────────────────────────────────────────────────────────────

print(f"Loading {MODEL_NAME} …")
print("First run will download ~15 GB model weights — please wait.")
print()

if USE_4BIT:
    from transformers import BitsAndBytesConfig
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_cfg,
        device_map="auto",
    )
else:
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype=DTYPE,
        device_map="auto" if DEVICE == "cuda" else None,
    )
    if DEVICE == "cpu":
        model = model.to(DEVICE)

model.eval()

# Processor: controls image token resolution.
# min/max_pixels: lower = faster, higher = more detail.
# 256*28*28 to 1280*28*28 is the recommended range for documents.
processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=256 * 28 * 28,
    max_pixels=1280 * 28 * 28,
)

print(f"\n✅ {MODEL_NAME} loaded successfully on {DEVICE}.")

## Cell 4 — Core OCR & Utility Functions

In [ ]:
# ── Default OCR system prompt ─────────────────────────────────────────────────
# This prompt is specifically tuned for Qwen2.5-VL to produce clean text output.
# It suppresses bounding-box / coordinate output which was the bug in v1.
DEFAULT_SYSTEM = (
    "You are an expert OCR engine. "
    "Extract ALL text from the image exactly as it appears — "
    "preserve line breaks, table structure, headings, and numbers. "
    "Do NOT output bounding boxes, coordinates, or JSON. "
    "Output plain text only."
)

DEFAULT_USER_PROMPT = "Extract all text from this image."


def prepare_image(image: Image.Image, max_size: tuple = (1920, 1920)) -> Image.Image:
    """Convert to RGB and resize to fit within max_size (keeps aspect ratio)."""
    image = image.convert("RGB")
    image.thumbnail(max_size, Image.Resampling.LANCZOS)
    return image


def extract_text(
    image: Image.Image,
    user_prompt: str = DEFAULT_USER_PROMPT,
    system_prompt: str = DEFAULT_SYSTEM,
    max_new_tokens: int = 2048,
) -> str:
    """
    Run Qwen2.5-VL OCR on a single PIL image.

    Key fix over v1:
    - Explicit system prompt suppressing coordinate output
    - Correct message format for Qwen2.5-VL (role=system + role=user)
    - inputs moved to correct device/dtype
    - torch.inference_mode() for speed and memory efficiency
    """
    messages = [
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image,
                    # Tell the model to use full resolution for document OCR
                    "resized_height": image.height,
                    "resized_width":  image.width,
                },
                {
                    "type": "text",
                    "text": user_prompt,
                },
            ],
        },
    ]

    # Build the text prompt using the chat template
    text_prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Extract image tensors from the message
    image_inputs, video_inputs = process_vision_info(messages)

    # Tokenise everything
    inputs = processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    # Move to device with correct dtype
    # NOTE: input_ids must stay int64, only float tensors get cast
    inputs = inputs.to(DEVICE)
    if "pixel_values" in inputs and DTYPE == torch.float16:
        inputs["pixel_values"] = inputs["pixel_values"].to(DTYPE)

    # Generate
    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,     # greedy — deterministic, best for OCR
            repetition_penalty=1.05,  # slight penalty to avoid repeated lines
        )

    # Trim the prompt tokens from the output
    trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    result = processor.batch_decode(
        trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    return result[0].strip()


# ── Keyword search ────────────────────────────────────────────────────────────
def search_keyword(text: str, query: str) -> str:
    """Case-insensitive search with match count and positions."""
    if not query.strip():
        return "⚠️  Please enter a keyword to search."
    matches = list(re.finditer(re.escape(query.strip()), text, flags=re.IGNORECASE))
    if matches:
        positions = ", ".join(str(m.start()) for m in matches)
        return f'✅ "{query}" found {len(matches)} time(s) at character position(s): {positions}.'
    return f'❌ "{query}" not found in extracted text.'


# ── Structure text ────────────────────────────────────────────────────────────
def structure_text(text: str) -> tuple:
    """Split into paragraphs, return (readable_str, json_dict)."""
    paragraphs = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]
    readable = "\n\n".join(
        f"Paragraph {i+1}:\n{p}" for i, p in enumerate(paragraphs)
    )
    json_data = {
        "total_paragraphs": len(paragraphs),
        "paragraphs": [
            {"id": i+1, "content": p, "word_count": len(p.split())}
            for i, p in enumerate(paragraphs)
        ],
    }
    return readable, json_data


print("✅ OCR functions ready.")

## Cell 5 — Pipeline (Image + PDF)

In [ ]:
def extract_from_pdf(pdf_path: str, user_prompt: str, system_prompt: str) -> str:
    """Run OCR page-by-page on a PDF and concatenate results."""
    if not PDF_SUPPORT:
        return "❌ PDF support unavailable. Run: !apt-get install -y poppler-utils"
    # dpi=250 gives enough resolution for tables and small fonts
    pages = convert_from_path(pdf_path, dpi=250)
    parts = []
    for i, page in enumerate(pages, start=1):
        img = prepare_image(page)
        text = extract_text(img, user_prompt=user_prompt, system_prompt=system_prompt)
        parts.append(f"── Page {i} ──\n{text}")
    return "\n\n".join(parts)


def ocr_pipeline(
    image_input,
    pdf_input,
    search_query: str,
    user_prompt: str,
    system_prompt: str,
):
    """
    Main Gradio handler:
    1. Accept image OR PDF  (PDF takes priority if both provided)
    2. Extract text with Qwen2.5-VL
    3. Keyword search
    4. Structure text + JSON
    """
    try:
        if image_input is None and pdf_input is None:
            msg = "⚠️  Please upload an image or a PDF."
            return msg, msg, msg, {}

        sys_p  = system_prompt.strip() if system_prompt.strip() else DEFAULT_SYSTEM
        user_p = user_prompt.strip()   if user_prompt.strip()   else DEFAULT_USER_PROMPT

        # Extract
        if pdf_input is not None:
            extracted = extract_from_pdf(pdf_input, user_p, sys_p)
        else:
            img = prepare_image(image_input)
            extracted = extract_text(img, user_prompt=user_p, system_prompt=sys_p)

        if not extracted.strip():
            extracted = "(No text detected.)"

        search_result          = search_keyword(extracted, search_query)
        structured, json_data  = structure_text(extracted)

        return extracted, search_result, structured, json_data

    except Exception as exc:
        err = f"❌ Error: {exc}\n\n{traceback.format_exc()}"
        return err, err, err, {"error": str(exc)}


print("✅ Pipeline ready.")

## Cell 6 — Gradio Interface

In [ ]:
with gr.Blocks(title="OCR — Qwen2.5-VL-7B", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🔍 OCR & Text Search — Qwen2.5-VL-7B-Instruct
    Upload an **image** (JPEG / PNG / WebP / BMP) **or a PDF** to extract and search text.\
    Supports English, Hindi, tables, and mixed-language documents.
    """)

    with gr.Row():
        # ── Inputs ───────────────────────────────────────────────────────────
        with gr.Column(scale=1):

            image_input = gr.Image(
                type="pil",
                label="📷 Upload Image (JPEG / PNG / WebP / BMP)",
            )
            pdf_input = gr.File(
                label="📄 Upload PDF  (takes priority over image if both provided)",
                file_types=[".pdf"],
                type="filepath",
            )
            search_query = gr.Textbox(
                label="🔎 Keyword to search in extracted text",
                placeholder="e.g.  SGPA  /  invoice  /  Bavya  …",
            )

            with gr.Accordion("⚙️  Advanced: Custom Prompts", open=False):
                user_prompt = gr.Textbox(
                    label="User prompt",
                    value=DEFAULT_USER_PROMPT,
                    lines=2,
                    info="Change this to focus the extraction, e.g. 'List all course codes and grades.'",
                )
                system_prompt = gr.Textbox(
                    label="System prompt",
                    value=DEFAULT_SYSTEM,
                    lines=4,
                    info="Controls the model's OCR behaviour. Edit only if you know what you are doing.",
                )

            with gr.Row():
                run_btn   = gr.Button("▶  Run OCR", variant="primary", scale=2)
                clear_btn = gr.ClearButton(
                    components=[image_input, pdf_input, search_query],
                    value="🗑  Clear",
                    scale=1,
                )

        # ── Outputs ──────────────────────────────────────────────────────────
        with gr.Column(scale=1):
            extracted_out = gr.Textbox(
                label="📝 Extracted Text",
                lines=14,
                show_copy_button=True,
            )
            search_out = gr.Textbox(
                label="🔎 Search Result",
                lines=2,
            )
            structured_out = gr.Textbox(
                label="🗂  Structured Text (paragraphs)",
                lines=10,
                show_copy_button=True,
            )
            json_out = gr.JSON(label="📦 JSON Output")

    gr.Markdown("""
    ---
    **Tips**
    - Grade cards, invoices, scanned PDFs → just click **Run OCR** with default settings.
    - For selective extraction (e.g. only grades) → open *Advanced* and edit the user prompt.
    - Multi-page PDFs are processed page by page; output is labelled `── Page N ──`.
    - Press **Enter** in the search box to re-search without re-running OCR.
    """)

    inputs  = [image_input, pdf_input, search_query, user_prompt, system_prompt]
    outputs = [extracted_out, search_out, structured_out, json_out]

    run_btn.click(fn=ocr_pipeline, inputs=inputs, outputs=outputs)
    search_query.submit(fn=ocr_pipeline, inputs=inputs, outputs=outputs)

print("✅ Gradio UI ready.")

## Cell 7 — Launch

In [ ]:
demo.launch(
    share=True,       # public gradio.live URL (expires in 72 h)
    show_error=True,  # surface Python tracebacks in the UI
    inbrowser=False,  # set True on local machine to auto-open browser
)